
## 1. System Cold Start Module Validation

The System Cold Start module implements mean-centred User-Based Collaborative Filtering.

It constructs a user-item rating matrix, calculates user similarities using cosine similarity, and predicts ratings using similarity-weighted deviations from neighbouring users' average ratings.

In this section, we verify that the module can:
- Load the processed training ratings and movie metadata.
- Train a collaborative filtering model.
- Generate recommendations for an existing user.
- Exclude movies that the user has already rated.


In [10]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.system_cold_start import (
    train_stage_model,
    recommend_stage_cf
)

print("System Cold Start module imported successfully!")


System Cold Start module imported successfully!


In [11]:

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Load processed datasets
# --------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

train_ratings = pd.read_csv(
    PROCESSED_DIR / "train_ratings.csv"
)

movie_features = pd.read_csv(
    PROCESSED_DIR / "movie_features.csv"
)

print("Training ratings:", train_ratings.shape)
print("Movie features:", movie_features.shape)

# --------------------------------------------------
# Train the System Cold Start CF model
# --------------------------------------------------

system_model = train_stage_model(train_ratings)

print("\nModel trained successfully!")
print("Number of users:", len(system_model["user_ids"]))
print("Number of movies:", len(system_model["movie_ids"]))

# --------------------------------------------------
# Select a user with sufficient rating history
# --------------------------------------------------

user_rating_counts = train_ratings["userID"].value_counts()

test_user_id = user_rating_counts.index[0]

print("\nSelected User ID:", test_user_id)
print("Ratings by selected user:", user_rating_counts.iloc[0])

# --------------------------------------------------
# Generate recommendations
# --------------------------------------------------

system_recommendations = recommend_stage_cf(
    model=system_model,
    user_id=test_user_id,
    movie_features=movie_features,
    top_n=10,
    k=20,
    min_support=2
)

display(system_recommendations)


Training ratings: (683631, 9)
Movie features: (10197, 27)

Model trained successfully!
Number of users: 2113
Number of movies: 9972

Selected User ID: 6757
Ratings by selected user: 2728


,movieID,title,genres,raw_prediction,predicted_rating,neighbor_support
0,8199,Ugetsu monogatari,Drama Thriller,4.729860,4.729860,2
1,7766,Kumonosu-jô,Action Drama Thriller War,4.642544,4.642544,2
2,25753,Greed,Drama,4.642544,4.642544,2
3,73,Les misérables,Drama War,4.642544,4.642544,2
4,5385,The Last Waltz,Documentary,4.586738,4.586738,2
5,1251,8½,Drama Fantasy,4.585856,4.585856,5
6,6727,Targets,Crime Thriller,4.476513,4.476513,2
7,5602,The Ladykillers,Comedy Crime,4.454000,4.454000,2
8,30803,Bin-jip,Drama Romance,4.391448,4.391448,2
9,3022,The General,Comedy War,4.382776,4.382776,6


## 2. Importing the Recommendation Components

The hybrid recommendation system combines three independently implemented
recommendation approaches:

1. System Cold Start — mean-centred User-Based Collaborative Filtering.
2. User Cold Start — TF-IDF-based recommendations from user preferences.
3. Item Cold Start — metadata-based recommendations for movies with no
   training interactions.

Each component generates its own recommendation list. These lists are
combined in the hybrid module using normalized scores.

In [12]:

# --------------------------------------------------
# Import all three recommendation components
# --------------------------------------------------

from src.system_cold_start import (
    train_stage_model,
    recommend_stage_cf
)

from src.user_cold_start import (
    build_tfidf_model as build_user_tfidf_model,
    recommend_for_new_user
)

from src.item_cold_start import (
    prepare_movie_features,
    build_tfidf_model as build_item_tfidf_model,
    build_movie_index,
    recommend_cold_start_movies
)

from src.hybrid import combine_recommendations

print("All recommendation modules imported successfully!")


All recommendation modules imported successfully!


## 3. Preparing Component Models

The processed training ratings and movie metadata are reused across
the recommendation components.

The System Cold Start model learns collaborative patterns from
historical user ratings.

The User Cold Start model builds TF-IDF representations of movie
metadata and uses selected favourite movies as onboarding preferences.

The Item Cold Start model uses movie metadata to recommend movies
that have no interactions in the training dataset.

In [13]:

# --------------------------------------------------
# 1. Prepare System Cold Start model
# --------------------------------------------------

system_model = train_stage_model(train_ratings)

# --------------------------------------------------
# 2. Prepare User Cold Start TF-IDF model
# --------------------------------------------------

user_tfidf_model = build_user_tfidf_model(movie_features)

# --------------------------------------------------
# 3. Prepare Item Cold Start TF-IDF model
# --------------------------------------------------

item_movie_features = prepare_movie_features(movie_features)

item_vectorizer, item_tfidf_matrix = build_item_tfidf_model(
    item_movie_features,
    feature_column="content"
)

item_movie_id_to_index = build_movie_index(item_movie_features)

# Identify movies with no interactions in the training dataset.
rated_movie_ids = set(train_ratings["movieID"].unique())

cold_start_movie_ids = [
    movie_id
    for movie_id in item_movie_features["movieID"].unique()
    if movie_id not in rated_movie_ids
]

print("All component models prepared successfully!")
print("Movies without training interactions:", len(cold_start_movie_ids))


All component models prepared successfully!
Movies without training interactions: 225


## 4. Generating Component Recommendations

A common user is selected to test the integration.

For the User Cold Start component, the user's highest-rated movies
are used as onboarding preferences.

For the Item Cold Start component, recommendations are restricted
to movies that have no interactions in the training dataset.

The collaborative filtering component uses the trained user-item
model and excludes movies already rated by the selected user.

In [14]:

# --------------------------------------------------
# Select the same test user
# --------------------------------------------------

test_user_id = train_ratings["userID"].value_counts().index[0]

user_history = train_ratings[
    train_ratings["userID"] == test_user_id
].copy()

already_rated = set(user_history["movieID"].unique())

print("Selected User:", test_user_id)
print("User's training ratings:", len(user_history))

# --------------------------------------------------
# A. System Cold Start recommendations
# --------------------------------------------------

cf_recommendations = recommend_stage_cf(
    model=system_model,
    user_id=test_user_id,
    movie_features=movie_features,
    top_n=20,
    k=20,
    min_support=2
)

# --------------------------------------------------
# B. User Cold Start recommendations
# --------------------------------------------------

# Select the user's highest-rated movies as onboarding preferences.
favourite_movies = (
    user_history
    .sort_values("rating", ascending=False)
    ["movieID"]
    .head(5)
    .tolist()
)

user_preferences = {
    "movies": favourite_movies
}

content_recommendations = recommend_for_new_user(
    preferences=user_preferences,
    movie_features=movie_features,
    top_n=20,
    exclude_movie_ids=already_rated,
    model=user_tfidf_model
)

# --------------------------------------------------
# C. Item Cold Start recommendations
# --------------------------------------------------

item_recommendations = recommend_cold_start_movies(
    user_id=test_user_id,
    train_ratings=user_history,
    movie_features=item_movie_features,
    cold_start_movie_ids=cold_start_movie_ids,
    tfidf_matrix=item_tfidf_matrix,
    movie_id_to_index=item_movie_id_to_index,
    top_n=20
)

# --------------------------------------------------
# Display component results
# --------------------------------------------------

print("\nSystem Cold Start:", len(cf_recommendations))
display(cf_recommendations.head())

print("\nUser Cold Start:", len(content_recommendations))
display(content_recommendations.head())

print("\nItem Cold Start:", len(item_recommendations))
display(item_recommendations.head())


Selected User: 6757
User's training ratings: 2728

System Cold Start: 20


,movieID,title,genres,raw_prediction,predicted_rating,neighbor_support
0,8199,Ugetsu monogatari,Drama Thriller,4.729860,4.729860,2
1,7766,Kumonosu-jô,Action Drama Thriller War,4.642544,4.642544,2
2,25753,Greed,Drama,4.642544,4.642544,2
3,73,Les misérables,Drama War,4.642544,4.642544,2
4,5385,The Last Waltz,Documentary,4.586738,4.586738,2



User Cold Start: 20


,movieID,title,genres,similarity
0,8228,The Maltese Falcon,Action Crime Drama Film-Noir Mystery,0.402755
1,1284,The Big Sleep,Crime Film-Noir Mystery,0.296322
2,910,Some Like It Hot,Comedy Crime,0.248086
3,4262,Scarface,Action Crime Drama,0.246310
4,954,Mr. Smith Goes to Washington,Drama,0.244382



Item Cold Start: 20


,movieID,title,recommendation_score
0,25942,Louisiana Story,0.165703
1,8450,Anthony Adverse,0.149402
2,51014,The Strange Case of Dr. Jekyll and Mr. Hyde,0.131628
3,26265,Dr. Phibes Rises Again,0.130666
4,32361,Come and Get It,0.128174


## 5. Hybrid Recommendation Generation

The three recommendation lists are combined using a weighted score
fusion strategy.

Before combination, scores from each recommendation component are
independently normalized to the range [0, 1].

The hybrid score is calculated using the available component scores
for each movie. This allows the system to recommend movies even when
a particular component does not generate a score for them.

The initial component weights are:

- System Cold Start: 50%
- User Cold Start: 30%
- Item Cold Start: 20%

In [15]:

# --------------------------------------------------
# Combine all three recommendation lists
# --------------------------------------------------

hybrid_recommendations = combine_recommendations(
    cf_recommendations=cf_recommendations,
    content_recommendations=content_recommendations,
    item_recommendations=item_recommendations,

    # Initial experimental weights
    weights=(0.50, 0.30, 0.20),

    top_n=10,

    # Exclude movies already rated by the selected user
    exclude_movie_ids=already_rated
)

print("Hybrid recommendations generated!")
print("Number of recommendations:", len(hybrid_recommendations))

display(hybrid_recommendations)


Hybrid recommendations generated!
Number of recommendations: 10


,movieID,title,genres,hybrid_score,cf_score,content_score,item_score
0,25942,Louisiana Story,NaN,1.000000,NaN,NaN,1.000000
1,8228,The Maltese Falcon,Action Crime Drama Film-Noir Mystery,1.000000,NaN,1.000000,NaN
2,8199,Ugetsu monogatari,Drama Thriller,1.000000,1.000000,NaN,NaN
3,73,Les misérables,Drama War,0.802685,0.802685,NaN,NaN
4,25753,Greed,Drama,0.802685,0.802685,NaN,NaN
5,7766,Kumonosu-jô,Action Drama Thriller War,0.802685,0.802685,NaN,NaN
6,8450,Anthony Adverse,NaN,0.754214,NaN,NaN,0.754214
7,5385,The Last Waltz,Documentary,0.676573,0.676573,NaN,NaN
8,1251,8½,Drama Fantasy,0.674580,0.674580,NaN,NaN
9,1284,The Big Sleep,Crime Film-Noir Mystery,0.529751,NaN,0.529751,NaN


## 6. Hybrid Integration Validation

The final recommendation list is checked for:

1. Duplicate movie IDs.
2. Movies already rated by the selected user.
3. Missing hybrid scores.
4. Correct ranking in descending hybrid-score order.
5. Availability of movie titles.

In [16]:

# --------------------------------------------------
# Validate hybrid recommendations
# --------------------------------------------------

print("Duplicate movie IDs:",
      hybrid_recommendations["movieID"].duplicated().sum())

print("Already-rated movies included:",
      hybrid_recommendations["movieID"].isin(already_rated).sum())

print("Missing hybrid scores:",
      hybrid_recommendations["hybrid_score"].isna().sum())

print("Missing titles:",
      hybrid_recommendations["title"].isna().sum())

print(
    "Sorted by hybrid score:",
    hybrid_recommendations["hybrid_score"].is_monotonic_decreasing
)

print("\nHybrid score summary:")
display(hybrid_recommendations["hybrid_score"].describe())


Duplicate movie IDs: 0
Already-rated movies included: 0
Missing hybrid scores: 0
Missing titles: 0
Sorted by hybrid score: True

Hybrid score summary:


count    10.000000
mean      0.804317
std       0.158440
min       0.529751
25%       0.695983
50%       0.802685
75%       0.950671
max       1.000000
Name: hybrid_score, dtype: float64

In [17]:
# Add movie metadata to item cold-start recommendations.
# This ensures every recommendation source has consistent movie details.

item_recommendations = item_recommendations.drop(
    columns=["title", "genres"],
    errors="ignore"
).merge(
    item_movie_features[["movieID", "title", "genres"]],
    on="movieID",
    how="left"
)

display(item_recommendations.head())

,movieID,recommendation_score,title,genres
0,25942,0.165703,Louisiana Story,Drama
1,8450,0.149402,Anthony Adverse,Adventure Romance
2,51014,0.131628,The Strange Case of Dr. Jekyll and Mr. Hyde,Drama Horror Sci-Fi Thriller
3,26265,0.130666,Dr. Phibes Rises Again,Adventure Comedy Horror Romance
4,32361,0.128174,Come and Get It,Drama


In [19]:
# Load the existing held-out test dataset.
# This must be the same test split used by the other experiments.

TEST_PATH = PROJECT_ROOT / "dataset" / "processed" / "test_ratings.csv"

test_ratings = pd.read_csv(TEST_PATH)

print("Test ratings shape:", test_ratings.shape)
print("Available columns:", test_ratings.columns.tolist())

display(test_ratings.head())

Test ratings shape: (171808, 9)
Available columns: ['userID', 'movieID', 'rating', 'date_day', 'date_month', 'date_year', 'date_hour', 'date_minute', 'date_second']


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second
0,75,2688,3.0,29,10,2006,23,22,26
1,75,165,4.5,29,10,2006,23,25,15
2,75,2700,4.5,29,10,2006,23,17,52
3,75,920,0.5,29,10,2006,23,17,8
4,75,1304,2.5,29,10,2006,23,16,56


## Comparative Evaluation of Recommendation Models

This experiment compares the recommendation quality of the individual
recommendation approaches and the integrated hybrid model.

The held-out test ratings are used only for evaluation. Ratings of 4.0
or above are considered relevant recommendations.

The evaluation metrics are:
- Precision@10
- Recall@10
- NDCG@10

The purpose is to examine whether combining recommendation sources
provides better results than using individual approaches.

In [20]:

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Obtain relevant movies from the held-out test set
# ---------------------------------------------------------

user_test_ratings = test_ratings[
    test_ratings["userID"] == test_user_id
].copy()

relevant_movies = set(
    user_test_ratings.loc[
        user_test_ratings["rating"] >= 4.0,
        "movieID"
    ]
)

print("Held-out ratings:", len(user_test_ratings))
print("Relevant movies:", len(relevant_movies))


# ---------------------------------------------------------
# 2. Define evaluation metrics
# ---------------------------------------------------------

def evaluate_recommendations(recommendation_ids, relevant_ids, k=10):

    recommendations = list(recommendation_ids)[:k]

    if not relevant_ids:
        return {
            "Precision@10": np.nan,
            "Recall@10": np.nan,
            "NDCG@10": np.nan
        }

    hits = [
        1 if movie_id in relevant_ids else 0
        for movie_id in recommendations
    ]

    precision = sum(hits) / k
    recall = sum(hits) / len(relevant_ids)

    # Discount relevant hits according to their ranking position.
    dcg = sum(
        hit / np.log2(position + 2)
        for position, hit in enumerate(hits)
    )

    ideal_hits = min(len(relevant_ids), k)

    idcg = sum(
        1 / np.log2(position + 2)
        for position in range(ideal_hits)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg
    }


# ---------------------------------------------------------
# 3. Evaluate individual recommendation approaches
# ---------------------------------------------------------

recommendation_sources = {
    "System CF": cf_recommendations,
    "User Content": content_recommendations,
    "Item Cold Start": item_recommendations,
    "Hybrid": hybrid_recommendations
}

evaluation_results = []

for model_name, recommendations in recommendation_sources.items():

    if recommendations is None or recommendations.empty:
        recommendation_ids = []
    else:
        recommendation_ids = (
            recommendations["movieID"].head(10).tolist()
        )

    metrics = evaluate_recommendations(
        recommendation_ids,
        relevant_movies,
        k=10
    )

    evaluation_results.append({
        "Model": model_name,
        **metrics
    })


# ---------------------------------------------------------
# 4. Display comparative results
# ---------------------------------------------------------

comparison_df = pd.DataFrame(evaluation_results)

display(
    comparison_df.round(4).sort_values(
        "NDCG@10",
        ascending=False
    )
)


Held-out ratings: 675
Relevant movies: 125


,Model,Precision@10,Recall@10,NDCG@10
1,User Content,0.4,0.032,0.3977
3,Hybrid,0.2,0.016,0.1331
0,System CF,0.1,0.008,0.0851
2,Item Cold Start,0.0,0.000,0.0000


In [21]:

# =========================================================
# MULTI-USER COMPARATIVE EVALUATION
# =========================================================

evaluation_users = (
    train_ratings.groupby("userID")
    .size()
    .loc[lambda x: x >= 20]
    .index
)

# Use a reproducible sample of up to 50 users.
evaluation_users = pd.Series(evaluation_users).sample(
    n=min(50, len(evaluation_users)),
    random_state=42
).tolist()

multi_user_results = []

for uid in evaluation_users:

    # Training history and held-out relevant movies
    user_history = train_ratings[
        train_ratings["userID"] == uid
    ].copy()

    user_test = test_ratings[
        test_ratings["userID"] == uid
    ]

    relevant = set(
        user_test.loc[
            user_test["rating"] >= 4.0, "movieID"
        ]
    )

    if not relevant:
        continue

    already_rated = set(user_history["movieID"])

    # 1. System Cold Start / Collaborative Filtering
    cf_recs = recommend_stage_cf(
        model=system_model,
        user_id=uid,
        movie_features=movie_features,
        top_n=20,
        k=20,
        min_support=2
    )

    # 2. User Cold Start / Content-Based
    favourite_movies = (
        user_history.loc[user_history["rating"] >= 4.0]
        .sort_values("rating", ascending=False)
        ["movieID"].head(5).tolist()
    )

    content_recs = recommend_for_new_user(
        preferences={"movies": favourite_movies},
        movie_features=movie_features,
        top_n=20,
        exclude_movie_ids=already_rated,
        model=user_tfidf_model
    )

    # 3. Item Cold Start
    item_recs = recommend_cold_start_movies(
        user_id=uid,
        train_ratings=user_history,
        movie_features=item_movie_features,
        cold_start_movie_ids=cold_start_movie_ids,
        tfidf_matrix=item_tfidf_matrix,
        movie_id_to_index=item_movie_id_to_index,
        top_n=20
    )

    # 4. Hybrid
    hybrid_recs = combine_recommendations(
        cf_recommendations=cf_recs,
        content_recommendations=content_recs,
        item_recommendations=item_recs,
        weights=(0.50, 0.30, 0.20),
        top_n=10,
        exclude_movie_ids=already_rated
    )

    sources = {
        "System CF": cf_recs,
        "User Content": content_recs,
        "Item Cold Start": item_recs,
        "Hybrid": hybrid_recs
    }

    for model_name, recs in sources.items():

        ids = (
            recs["movieID"].head(10).tolist()
            if recs is not None and not recs.empty
            else []
        )

        metrics = evaluate_recommendations(
            ids, relevant, k=10
        )

        multi_user_results.append({
            "userID": uid,
            "Model": model_name,
            **metrics
        })

# Store individual user results
multi_user_df = pd.DataFrame(multi_user_results)

# Macro-average metrics across evaluated users
multi_user_summary = (
    multi_user_df.groupby("Model")[
        ["Precision@10", "Recall@10", "NDCG@10"]
    ]
    .mean()
    .sort_values("NDCG@10", ascending=False)
)

print("Users evaluated:", multi_user_df["userID"].nunique())
display(multi_user_summary.round(4))


Users evaluated: 50


,Precision@10,Recall@10,NDCG@10
Model,,,
User Content,0.174,0.0700,0.2083
Hybrid,0.076,0.0304,0.0916
System CF,0.076,0.0267,0.0771
Item Cold Start,0.000,0.0000,0.0000


In [27]:

# =========================================================
# SPARSITY-AWARE HYBRID EVALUATION
# =========================================================

adaptive_results = []

for uid in evaluation_users:

    # Obtain this user's training history
    user_history = train_ratings[
        train_ratings["userID"] == uid
    ].copy()

    user_test = test_ratings[
        test_ratings["userID"] == uid
    ]

    relevant = set(
        user_test.loc[
            user_test["rating"] >= 4.0,
            "movieID"
        ]
    )

    if not relevant:
        continue

    already_rated = set(user_history["movieID"])

    # Generate the same three recommendation sources
    cf_recs = recommend_stage_cf(
        model=system_model,
        user_id=uid,
        movie_features=movie_features,
        top_n=20,
        k=20,
        min_support=2
    )

    favourite_movies = (
        user_history.loc[user_history["rating"] >= 4.0]
        .sort_values("rating", ascending=False)
        ["movieID"].head(5).tolist()
    )

    content_recs = recommend_for_new_user(
        preferences={"movies": favourite_movies},
        movie_features=movie_features,
        top_n=20,
        exclude_movie_ids=already_rated,
        model=user_tfidf_model
    )

    item_recs = recommend_cold_start_movies(
        user_id=uid,
        train_ratings=user_history,
        movie_features=item_movie_features,
        cold_start_movie_ids=cold_start_movie_ids,
        tfidf_matrix=item_tfidf_matrix,
        movie_id_to_index=item_movie_id_to_index,
        top_n=20
    )

    # Calculate adaptive weights from available user history
    user_rating_count = len(user_history)

    adaptive_weights = get_sparsity_aware_weights(
        user_rating_count
    )

    # Generate sparsity-aware hybrid recommendations
    adaptive_hybrid = combine_recommendations(
        cf_recommendations=cf_recs,
        content_recommendations=content_recs,
        item_recommendations=item_recs,
        weights=adaptive_weights,
        top_n=10,
        exclude_movie_ids=already_rated
    )

    recommendation_ids = (
        adaptive_hybrid["movieID"].tolist()
        if not adaptive_hybrid.empty
        else []
    )

    metrics = evaluate_recommendations(
        recommendation_ids,
        relevant,
        k=10
    )

    adaptive_results.append({
        "userID": uid,
        "User_Ratings": user_rating_count,
        "CF_Weight": adaptive_weights[0],
        "Content_Weight": adaptive_weights[1],
        "Item_Weight": adaptive_weights[2],
        **metrics
    })


# Store adaptive evaluation
adaptive_df = pd.DataFrame(adaptive_results)

# Compare with existing fixed-weight hybrid
fixed_summary = (
    multi_user_df[
        multi_user_df["Model"] == "Hybrid"
    ][["Precision@10", "Recall@10", "NDCG@10"]]
    .mean()
)

adaptive_summary = adaptive_df[
    ["Precision@10", "Recall@10", "NDCG@10"]
].mean()

hybrid_comparison = pd.DataFrame([
    {
        "Model": "Fixed Weight Hybrid",
        **fixed_summary.to_dict()
    },
    {
        "Model": "Sparsity-Aware Hybrid",
        **adaptive_summary.to_dict()
    }
])

print("Users evaluated:", len(adaptive_df))
display(hybrid_comparison.round(4))


Users evaluated: 50


,Model,Precision@10,Recall@10,NDCG@10
0,Fixed Weight Hybrid,0.076,0.0304,0.0916
1,Sparsity-Aware Hybrid,0.080,0.0340,0.0950


In [32]:
# Save sparsity-aware experiment results

adaptive_df.to_csv(
    results_dir / "sparsity_aware_hybrid_results.csv",
    index=False
)

hybrid_comparison.to_csv(
    results_dir / "fixed_vs_adaptive_hybrid.csv",
    index=False
)

print("Sparsity-aware hybrid results saved.")

Sparsity-aware hybrid results saved.


In [33]:
# Save multi-user evaluation results

results_dir = PROJECT_ROOT / "outputs" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

multi_user_df.to_csv(
    results_dir / "hybrid_multi_user_evaluation.csv",
    index=False
)

multi_user_summary.to_csv(
    results_dir / "hybrid_model_comparison.csv"
)

print("Evaluation results saved successfully.")

Evaluation results saved successfully.


## Comparative Evaluation Results

The four recommendation approaches were evaluated across 50 users
using the same held-out test dataset.

User Content achieved the highest Precision@10, Recall@10 and
NDCG@10 among the evaluated approaches.

The initial Hybrid model did not outperform the individual
User Content approach. This indicates that combining multiple
recommendation sources does not necessarily improve recommendation
quality, particularly when the sources have different score
distributions and recommendation strengths.

The Item Cold Start approach produced no relevant recommendations
in the evaluated top-10 lists. This highlights the difficulty of
recommending globally unseen items based on limited metadata and
user preferences.

These results are based on the selected evaluation sample and
should not be interpreted as universal performance guarantees.

In [34]:
# Examine rating availability across users

user_rating_counts = train_ratings.groupby("userID").size()

print("Minimum ratings:", user_rating_counts.min())
print("Maximum ratings:", user_rating_counts.max())
print("Average ratings:", user_rating_counts.mean())
print("Median ratings:", user_rating_counts.median())

display(user_rating_counts.describe())

Minimum ratings: 16
Maximum ratings: 2728
Average ratings: 323.53573118788455
Median ratings: 218.0


count    2113.000000
mean      323.535731
std       336.237671
min        16.000000
25%        85.000000
50%       218.000000
75%       441.000000
max      2728.000000
dtype: float64

In [35]:
import importlib
import src.hybrid

importlib.reload(src.hybrid)

from src.hybrid import (
    combine_recommendations,
    get_sparsity_aware_weights
)

print("Hybrid module reloaded successfully!")

Hybrid module reloaded successfully!


In [36]:
for count in [20, 75, 218, 400, 700]:
    print(
        f"{count} ratings:",
        get_sparsity_aware_weights(count)
    )

20 ratings: (0.1, 0.7, 0.2)
75 ratings: (0.2, 0.6, 0.2)
218 ratings: (0.4, 0.4, 0.2)
400 ratings: (0.55, 0.3, 0.15)
700 ratings: (0.65, 0.25, 0.1)


In [37]:
# Save final hybrid comparison and adaptive results

hybrid_comparison.to_csv(
    results_dir / "fixed_vs_adaptive_hybrid.csv",
    index=False
)

adaptive_df.to_csv(
    results_dir / "sparsity_aware_hybrid_results.csv",
    index=False
)

# Save the overall four-model comparison
final_comparison = pd.DataFrame([
    {
        "Model": "User Content",
        "Precision@10": 0.174,
        "Recall@10": 0.0700,
        "NDCG@10": 0.2083
    },
    {
        "Model": "System CF",
        "Precision@10": 0.076,
        "Recall@10": 0.0267,
        "NDCG@10": 0.0771
    },
    {
        "Model": "Item Cold Start",
        "Precision@10": 0.000,
        "Recall@10": 0.0000,
        "NDCG@10": 0.0000
    },
    {
        "Model": "Fixed Weight Hybrid",
        "Precision@10": 0.076,
        "Recall@10": 0.0304,
        "NDCG@10": 0.0916
    },
    {
        "Model": "Sparsity-Aware Hybrid",
        "Precision@10": 0.080,
        "Recall@10": 0.0340,
        "NDCG@10": 0.0950
    }
])

final_comparison.to_csv(
    results_dir / "final_model_comparison.csv",
    index=False
)

print("All final comparison files saved.")

All final comparison files saved.


### Sparsity-Aware Hybrid Integration

To integrate the four recommendation components, a sparsity-aware hybrid recommendation strategy was developed. The data sparsity experiment demonstrated that recommendation accuracy varies with the availability of user-item interactions.

Based on this observation, an adaptive weighting mechanism was introduced into the hybrid recommendation module. The mechanism uses the number of ratings available for a user to determine the contribution of collaborative filtering, user-based content recommendations, and item cold-start recommendations.

Users with limited rating histories receive greater weight from content-based recommendations, while users with richer interaction histories receive greater weight from collaborative filtering.

The proposed approach was evaluated against the initial fixed-weight hybrid model using the same sample of 50 users and held-out test data.

The fixed-weight hybrid achieved Precision@10 of 0.0760, Recall@10 of 0.0304, and NDCG@10 of 0.0916. The sparsity-aware hybrid achieved Precision@10 of 0.0800, Recall@10 of 0.0340, and NDCG@10 of 0.0950.

The results indicate modest improvements across all three evaluation metrics. This suggests that adapting recommendation weights according to user interaction availability can improve the initial hybrid recommendation strategy.

However, the adaptive hybrid did not outperform the standalone user content model. Further optimization of fusion weights, candidate generation, and evaluation across a larger user sample would be required to establish broader performance improvements.